# E2 repair: exact failed-frame diagnosis, then unfinished sequences
First replay seq5 frame137 as E0 only, recording the legacy mutation and the corrected pre-gate memory-read control. Proceed only if it passes. Then run all 887 eligible targets in seq5, seq6 and seq8–12 with mandatory online W&B and fresh checks at every target. Previously completed seq2–4 remain untouched in the prior output. This is a repaired E2 component, not the full study.

In [ ]:
print('[notebook] kernel_started: restoration diagnosis then E2 continuation', flush=True)
import json, os, urllib.request
from pathlib import Path
REVISION = 'c81e7af'
job = dict(stage='E2', dataset='polypgen', split='test', sequences=['seq5', 'seq6', 'seq8', 'seq9', 'seq10', 'seq11', 'seq12'], mode='single_step', intervention='attention_mask', memory_condition='c0', band=1, thresholds=[0.1, 0.9], label_ids=[1])
os.environ['ADESEG_REVISION'] = REVISION
os.environ['REGION_E1_RUN_ID'] = '7orfgsk2'
url = 'https://raw.githubusercontent.com/aCoderChild/AdeSEG/' + REVISION + '/kaggle/region-tokens/runner.py'
print('[notebook] fetching runner revision=' + REVISION, flush=True)
with urllib.request.urlopen(url, timeout=60) as response: source = response.read().decode('utf-8')
os.environ.update(REGION_STAGE='E0', REGION_E0_SEQUENCE='seq5', REGION_E0_TARGET_FRAME='137', REGION_OUTPUT_NAME='region_tokens_restore_diag_seq5_137_v2')
exec(compile(source, 'region_tokens_runner.py', 'exec'), {'__name__': '__main__'})
checks = [json.loads(line) for line in Path('/kaggle/working/region_tokens_restore_diag_seq5_137_v2/E0/sanity.jsonl').read_text().splitlines() if line]
assert len(checks) == 1 and checks[0]['frame'] == 137 and checks[0]['restore_control_version'] == 2 and checks[0]['passed'], 'Exact failed-frame GPU diagnosis did not pass; stop before E2'
print('[notebook] failed-frame diagnosis passed', json.dumps(checks[0]), flush=True)
os.environ.pop('REGION_E0_TARGET_FRAME', None)
os.environ.pop('REGION_E0_SEQUENCE', None)
os.environ.update(REGION_STAGE='study_shard', REGION_JOB=json.dumps(job), REGION_OUTPUT_NAME='region_tokens_e2_polypgen_primary_batch01_r3')
print('[notebook] starting unfinished E2 units', json.dumps(job), flush=True)
exec(compile(source, 'region_tokens_runner.py', 'exec'), {'__name__': '__main__'})
